In [ ]:
# This code extracts necessary information from the data created in preprocessing file.
# This code is specific to 2AFC - contrast task for the sideBiasLateralisation project.

%reload_ext autoreload
%autoreload 2

#load libraries
import glob
import os
import json
import pandas as pd
from datetime import datetime
import numpy as np
import main_funcs as mfun
import plot_funcs as pfun
import utils_funcs as utils 
import LakLabAnalysis.Utility.utils_funcs as utils_laklab
import LakLabAnalysis.Utility.commonPlot_funcs as cpfun # utils is from Vape - catcher file: 
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
import pickle



In [ ]:
# Plotting params
pfun.set_figure()

## Parameters
fps = 30
fRate = 1000/fps
responsiveness_test_duration = 1000.0 #in ms 
pre_frames    = 2000.0# in ms
pre_frames    = int(np.ceil(pre_frames/fRate))
post_frames   = 6000.0 # in ms
post_frames   = int(np.ceil(post_frames/fRate))
analysisWindowDur = 750 # in ms
analysisWindowDur = int(np.ceil(analysisWindowDur/fRate))
simulationDur_ms = 2000.0 # in ms 
simulationDur  = int(np.ceil(simulationDur_ms/fRate))
duration ='5'

pd.set_option('mode.chained_assignment', None)


In [ ]:
# You should always read the saved info! Get the list to extract the files for further analysis
repo_analysis_path = r'C:/Users/Huriye/Documents/code/sideBiasLateralisation/analysis'
info = pickle.load(open(os.path.join(repo_analysis_path, 'sideBiasLateralisation_info.pkl'), 'rb'))
print('Total Session fits the selection: ' +  str(info.recordingList.shape[0]))


In [ ]:
# Extract necessary files for each session
use_responsive_only = False  # Change to False to use all neurons // Option to use only responsive neurons

for ind, recordingDate in enumerate(info.recordingList.recordingDate):
      try: #if ind> -100: # 
         print(str(ind) + ': Creating: ' + info.recordingList.analysispathname[ind])
         #Create a huge dictionary with all cells and parameters for each cell
         pathname = info.recordingList.analysispathname[ind]
         # create the folder if it does not exist
         if not os.path.exists(pathname):
            os.makedirs(pathname)

         ########## GET BEHAVIOURAL OUTPUT
         # Lets load the beh file with excluded trials do a little cleaning here for trials/sessions
         behData, index_clean, _ = mfun.clean_session_behavior(info.recordingList.analysispathname[ind],
                                                               info.recordingList.sessionName[ind])
         
         visTimes    = behData['stimulusOnsetTime'] + behData['trialOffsets']
         rewardTimes = behData['rewardTime'] + behData['trialOffsets']
         choiceTimes = behData['choiceStartTime'] + behData['trialOffsets']
      
         choiceCompleteTimes = behData['choiceCompleteTime'] + behData['trialOffsets']
         diff_time = np.nanmean(rewardTimes -choiceTimes)
         nan_indices = np.isnan(rewardTimes) # unrewarded trials
         rewardTimes[nan_indices] = choiceCompleteTimes[nan_indices] + diff_time
         
         # Get the stim start times 
         filenameTXT = os.path.join(info.recordingList.path[ind],'twoP') +'\*_imaging_frames.txt'
         filenameTXT= [f for f in glob.glob(filenameTXT)]    
         frame_clock = pd.read_csv(filenameTXT[0],  header= None)
            
         stimFrameTimes    = utils.stim_start_frame_Dual2Psetup (frame_clock, visTimes)
         rewardFrameTimes  = utils.stim_start_frame_Dual2Psetup (frame_clock, rewardTimes)
         choiceFrameTimes  = utils.stim_start_frame_Dual2Psetup (frame_clock, choiceTimes)

      ########## GET CALCIUM IMAGING DATA
         imData = pd.read_pickle (pathname +'imaging-data.pkl')
         fluRaw    = imData['flu']
         stat      = imData['stat']

         # clean detrended traces
         flu = utils.clean_traces(fluRaw)
         
         #flu_smoothed = utils_laklab.preprocess_flu(fluR, smooth_method='savgol', do_zscore=False, smooth_first=True)      
         # We use a very low threshold for cell selection - lets clean small ROIs and low SNR cells

         # filtered_cells = utils.snr_filter_cells(flu, snr_thresh='auto', info=info, 
         #                                        blockName=info.recordingList.blockName[ind], 
         #                                        plot=True, 
         #                                        savepath=info.recordingList.analysispathname[ind],
         #                                        cell_radiusThreshold = 5)

         filtered_cells = np.arange(flu.shape[0]) # if we want to use all cells without filtering
         flu = flu[filtered_cells,:]
         stat = [stat[i] for i in filtered_cells]
         # print the number of cells after filtering
         print(f'Number of cells after SNR filtering: {len(stat)}')

         flu_zscored = zscore(flu, axis=1)# Z-score for each neuron (across time)
         flu_normalised = zscore(flu)

         # Lets plot the FOV with selected cells again
         s2p_path = os.path.join(info.recordingList.path[ind], 'TwoP', f"{info.recordingList.recordingDate[ind]}_t-001", 'suite2p', 'plane0')
         isCell = np.load(os.path.join(s2p_path, 'iscell.npy'), allow_pickle=True)
         isCell[:,0] = 1 # Set all cells to 1 to plot all cells that are in the stat file (after filtering)
         ops = np.load(os.path.join(s2p_path, 'ops.npy'), allow_pickle=True)
         ops = ops.item()

         # Load the suite2p
         filtered_isCell = isCell[filtered_cells,:]

         # pfun.create_FOV_withSelectedCells(filtered_isCell, ops, s2p_path, 
         #                        info.recordingList.analysispathname[ind], savename='SNRfiltered_', stat=stat)
         # # print the number of cells after filtering
         # print(f'Number of cells plotted after SNR filtering: {len(filtered_isCell)}')
         
      ########## CREATE INTERESTED TRIAL TYPES
         tTypesName, tTypes = mfun.build_trial_types(behData,
                                                   info.recordingList.imagingTiffFileNames[ind])

         ########## CREATE TRIAL TYPE SPECIFIC TRACES
         # --- FULL TRACE ---
         dffTrace_reward, dffTrace_mean_reward = {}, {}
         dffTrace_stimuli, dffTrace_mean_stimuli = {}, {}
         dffTrace_choice, dffTrace_mean_choice = {}, {}
         dff_mean_stimuli, dff_mean_reward, dff_mean_choice = {}, {}, {}
         dffTrace_mean_stimuli_baseline, dffTrace_mean_reward_baseline, dffTrace_mean_choice_baseline = {}, {}, {}

         for indx, t in enumerate ([tTypesName[0]]):
            # For reward
            selected_indices = [rewardFrameTimes[i] for i in np.where((tTypes[indx]==True)& (~np.isnan(rewardFrameTimes)))[0]]  
            selected_indices = [value for value in selected_indices if value == value]
            print(len(selected_indices))
            dffTrace_reward[t] = utils.flu_splitter(flu,selected_indices, pre_frames, post_frames)  # Cell x time x trial
            # dffTrace_mean_reward[t] = np.mean(dffTrace_reward[t],2) if len(selected_indices)>2 else None # Cell x time
            # dff_mean_reward[t] = np.nanmean(dffTrace_mean_reward[t][:, (pre_frames): (pre_frames + analysisWindowDur)],1) if len(selected_indices)>2 else None
            # baseline  = np.nanmean(dffTrace_mean_reward[t][:, (pre_frames - int(np.ceil(1000.0/fRate))): pre_frames],axis=1, keepdims=True) if len(selected_indices)>2 else None
            # dffTrace_mean_reward_baseline[t] = (dffTrace_mean_reward[t] - baseline)  if len(selected_indices)>2 else None

            # For stimuli
            selected_indices = [stimFrameTimes[i] for i in np.where(tTypes[indx]==True)[0]]
            selected_indices = [value for value in selected_indices if value == value]
            dffTrace_stimuli[t] = utils.flu_splitter(flu, selected_indices, pre_frames, post_frames) # Cell x time x trial 
            # dffTrace_mean_stimuli[t] = np.mean(dffTrace_stimuli[t],2) if len(selected_indices)>2 else None # Cell x time
            # dff_mean_stimuli[t] = np.nanmean(dffTrace_mean_stimuli[t][:, (pre_frames): (pre_frames + analysisWindowDur)],1) if len(selected_indices)>2 else None
            # baseline  = np.nanmean(dffTrace_mean_stimuli[t][:, (pre_frames - int(np.ceil(1000.0/fRate))): pre_frames],axis=1, keepdims=True) if len(selected_indices)>2 else None
            # dffTrace_mean_stimuli_baseline[t] = (dffTrace_mean_stimuli[t] - baseline) if len(selected_indices)>2 else None

            # For choice
            selected_indices = [choiceFrameTimes[i] for i in np.where((tTypes[indx]==True)& (~np.isnan(choiceFrameTimes)))[0]] 
            selected_indices = [value for value in selected_indices if value == value]
            dffTrace_choice[t] = utils.flu_splitter(flu, selected_indices, pre_frames, post_frames) # Cell x time x trial
            # dffTrace_mean_choice[t] = np.mean(dffTrace_choice[t],2) if len(selected_indices)>2 else None # Cell x time
            # dff_mean_choice[t] = np.nanmean(dffTrace_mean_choice[t][:, (pre_frames): (pre_frames + analysisWindowDur)],1) if len(selected_indices)>2 else None
            # baseline  = np.nanmean(dffTrace_mean_choice[t][:, (pre_frames - int(np.ceil(1000.0/fRate))): pre_frames],axis=1, keepdims=True) if len(selected_indices)>2 else None
            # dffTrace_mean_choice_baseline[t] = (dffTrace_mean_choice[t] - baseline) if len(selected_indices)>2 else None


           # --- Z-SCORED ---
         dffTrace_reward_z, dffTrace_mean_reward_z = {}, {}
         dffTrace_stimuli_z, dffTrace_mean_stimuli_z = {}, {}
         dffTrace_choice_z, dffTrace_mean_choice_z = {}, {}
         dff_mean_stimuli_z, dff_mean_reward_z, dff_mean_choice_z = {}, {}, {}
         
         for indx, t in enumerate ([tTypesName[0]]):
            # Reward
            selected_indices = [rewardFrameTimes[i] for i in np.where((tTypes[indx]==True)& (~np.isnan(rewardFrameTimes)))[0]]  
            selected_indices = [value for value in selected_indices if value == value]
            dffTrace_reward_z[t] = utils.flu_splitter(flu_zscored, selected_indices, pre_frames, post_frames)
            # dffTrace_mean_reward_z[t] = np.mean(dffTrace_reward_z[t],2) if len(selected_indices)>2 else None
            # dff_mean_reward_z[t] = np.nanmean(dffTrace_mean_reward_z[t][:, (pre_frames): (pre_frames + analysisWindowDur)],1) if len(selected_indices)>2 else None
            
            # Stimuli
            selected_indices = [stimFrameTimes[i] for i in np.where(tTypes[indx]==True)[0]]
            selected_indices = [value for value in selected_indices if value == value]
            dffTrace_stimuli_z[t] = utils.flu_splitter(flu_zscored, selected_indices, pre_frames, post_frames)
            # dffTrace_mean_stimuli_z[t] = np.mean(dffTrace_stimuli_z[t],2) if len(selected_indices)>2 else None
            # dff_mean_stimuli_z[t] = np.nanmean(dffTrace_mean_stimuli_z[t][:, (pre_frames): (pre_frames + analysisWindowDur)],1) if len(selected_indices)>2 else None

            # Choice
            selected_indices = [choiceFrameTimes[i] for i in np.where((tTypes[indx]==True)& (~np.isnan(choiceFrameTimes)))[0]] 
            selected_indices = [value for value in selected_indices if value == value]
            dffTrace_choice_z[t] = utils.flu_splitter(flu_zscored, selected_indices, pre_frames, post_frames)
            # dffTrace_mean_choice_z[t] = np.mean(dffTrace_choice_z[t],2) if len(selected_indices)>2 else None
            # dff_mean_choice_z[t] = np.nanmean(dffTrace_mean_choice_z[t][:, (pre_frames): (pre_frames + analysisWindowDur)],1) if len(selected_indices)>2 else None

      #
      ########## CHECK RESPONSIVENESS with Ttest
      # This is too risky to exclude cells - as it may remove important information about the neural dynamics
      # However it is good to look for percentages of responsive cells for each trial type
         # pvals, pvalsWilcoxon = {}, {}

         # for indx, t in enumerate(tTypesName):
         #    trialInd = np.where(tTypes[indx]==True)[0]
         #    if len(trialInd)>1:
         #       _, _, pval = utils.test_responsive  (flu, frame_clock, np.array(stimFrameTimes)[trialInd], pre_frames=int(np.ceil(responsiveness_test_duration/fRate)), 
         #                                                    post_frames=int(np.ceil(responsiveness_test_duration/fRate)),
         #                                                    testType ='ttest')
         #       pvals[t] = pval
         #       _, _, pval = utils.test_responsive  (flu, frame_clock, np.array(stimFrameTimes)[trialInd], pre_frames=int(np.ceil(responsiveness_test_duration/fRate)), 
         #                                                    post_frames=int(np.ceil(responsiveness_test_duration/fRate)),
         #                                                    testType ='wilcoxon')
         #       pvalsWilcoxon[t] = pval
         # # print the number of total number of p>0.05 cells 
         # for t in tTypesName:
         #    if t in pvals:
         #       print(f'Trial type: {t}, Percentage of non-responsive cells (p>0.05) with t-test: {np.sum(pvals[t]>0.05)/len(pvals[t])*100:.2f}%')
         #    if t in pvalsWilcoxon:
         #       print(f'Trial type: {t}, Percentage of non-responsive cells (p>0.05) with Wilcoxon test: {np.sum(pvalsWilcoxon[t]>0.05)/len(pvalsWilcoxon[t])*100:.2f}%')

         # ########## CHECK RESPONSIVENESS with ZetaTest
         # DO WE REALLY NEED THIS? CODE IS NOT WRITTEN YET
         
         ########## CHECK RESPONSIVENESS with SNR 
         # DO WE REALLY NEED THIS? CODE IS NOT WRITTEN YET

         ########## LETS SAVE VARIABLES

         # Determine subfolder based on use_responsive_only
         subfolder = 'responsive_neurons' if use_responsive_only else 'all_neurons'
         subfolder_path = os.path.join(pathname, subfolder)
         os.makedirs(subfolder_path, exist_ok=True)

         # Save params in the appropriate subfolder
         filenameINFO = os.path.join(subfolder_path, 'imaging-dffTraceAll.pkl')
         print('Saving: '+ filenameINFO)
         with open(filenameINFO, 'wb') as f:
            pickle.dump([dffTrace_reward,dffTrace_stimuli, dffTrace_choice] , f)

         # filenameINFO = os.path.join(subfolder_path, 'imaging-dffTrace_mean.pkl')
         # print('Saving: '+ filenameINFO)
         # with open(filenameINFO, 'wb') as f:
         #    pickle.dump([dffTrace_mean_reward,
         #                dffTrace_mean_stimuli,
         #                dffTrace_mean_choice ] , f)

         # filenameINFO = os.path.join(subfolder_path, 'imaging-dff_mean.pkl')
         # print('Saving: '+ filenameINFO)
         # with open(filenameINFO, 'wb') as f:
         #    pickle.dump([dff_mean_reward,
         #                dff_mean_stimuli,
         #                dff_mean_choice ] , f)

         # Save z-scored data in the same subfolder
         filenameINFO = os.path.join(subfolder_path, 'imaging-dffTrace_zscoredAll.pkl')
         print('Saving: '+ filenameINFO)
         with open(filenameINFO, 'wb') as f:
            pickle.dump([dffTrace_reward_z, dffTrace_stimuli_z, dffTrace_choice_z], f)

         # filenameINFO = os.path.join(subfolder_path, 'imaging-dffTrace_mean_zscored.pkl')
         # print('Saving: '+ filenameINFO)
         # with open(filenameINFO, 'wb') as f:
         #    pickle.dump([dffTrace_mean_reward_z, dffTrace_mean_stimuli_z, dffTrace_mean_choice_z], f)

         # filenameINFO = os.path.join(subfolder_path, 'imaging-dff_mean_zscored.pkl')
         # print('Saving: '+ filenameINFO)
         # with open(filenameINFO, 'wb') as f:
         #    pickle.dump([dff_mean_reward_z, dff_mean_stimuli_z, dff_mean_choice_z], f)

         # filenameINFO = os.path.join(subfolder_path, 'imaging-dff_mean_baseline.pkl')
         # print('Saving: '+ filenameINFO)
         # with open(filenameINFO, 'wb') as f:
         #    pickle.dump([dffTrace_mean_reward_baseline, dffTrace_mean_stimuli_baseline, dffTrace_mean_choice_baseline], f)

         # ########## LETS CREATE SOME HEATMAPS
         # colormap = 'viridis'
         # selectedSession = 'WithinSession'
         
         # savefigname = 'SessionComparison-heatmap-reward_' + str(duration[0]) + 'sec'
         # analysis_params = ['Rewarded', 'Unrewarded']
         # pfun.heatmap_sessions(dffTrace_mean_reward_baseline, analysis_params, colormap,
         #                      selectedSession, duration, savefigname, subfolder_path) 
         # plt.close()
                        
         # savefigname = 'SessionComparison-heatmap-choice_' + str(duration[0]) + 'sec'
         # analysis_params = ['Left Choices','Right Choices']
         # pfun.heatmap_sessions(dffTrace_mean_choice_baseline, analysis_params, colormap,
         #                      selectedSession, duration, savefigname, subfolder_path) 
         # plt.close()

         # savefigname = 'SessionComparison-heatmap-stimuli_' + str(duration[0]) + 'sec'
         # analysis_params = ['0.0625','0.125', '0.25', '0.5', '0']
         # pfun.heatmap_sessions(dffTrace_mean_stimuli_baseline, analysis_params, colormap,
         #                      selectedSession, duration, savefigname, subfolder_path) 
         # plt.close()


         # ########## LETS CREATE SOME LINEPLOTS for Average Traces
         # zscoreRun = False 
         # baseline_subtract = [-1.0, 0.0]  # Baseline window: 1 second before stimulus onset

         # # Get animal number and session info
         # session_name = info.recordingList.sessionName[ind]
         # title_prefix = f'{session_name}'

         # colormap = ['red', 'black']
         # savefigname = 'SessionComparison-mean-reward_' + str(duration[0]) + 'sec'
         # analysis_params = ['Rewarded', 'Unrewarded']
         # pfun.lineplot_sessions(dffTrace_mean_reward_baseline, analysis_params, colormap,
         #                      duration, zscoreRun, savefigname, subfolder_path, baseline_subtract, 
         #                      title=f'{title_prefix} - Rewarded vs Unrewarded / reward aligned') 
         # plt.close()

         # colormap = ['blue', 'red']
         # savefigname = 'SessionComparison-mean-choice_' + str(duration[0]) + 'sec'
         # analysis_params = ['Left Choices','Right Choices']
         # pfun.lineplot_sessions(dffTrace_mean_choice_baseline, analysis_params, colormap,
         #                      duration, zscoreRun, savefigname, subfolder_path, baseline_subtract,
         #                      title=f'{title_prefix} - Choice side / choice aligned')
         # plt.close()

         # colormap = 'viridis'
         # savefigname = 'SessionComparison-mean-stimuli-rewarded-contrasts_' + str(duration[0]) + 'sec'
         # analysis_params = ['0.5 Rewarded', '0.25 Rewarded','0.125 Rewarded','0.0625 Rewarded','0 Rewarded']
         # pfun.lineplot_sessions(dffTrace_mean_stimuli_baseline, analysis_params, colormap,
         #                      duration, zscoreRun, savefigname, subfolder_path, baseline_subtract,
         #                      title=f'{title_prefix} - Rewarded stimuli / stimulus aligned') 
         # plt.close()

         # colormap = 'viridis'
         # savefigname = 'SessionComparison-mean-reward-rewarded-contrasts_' + str(duration[0]) + 'sec'
         # analysis_params = ['0.5 Rewarded', '0.25 Rewarded','0.125 Rewarded','0.0625 Rewarded','0 Rewarded']
         # pfun.lineplot_sessions(dffTrace_mean_reward_baseline, analysis_params, colormap,
         #                      duration, zscoreRun, savefigname, subfolder_path, baseline_subtract,
         #                      title=f'{title_prefix} - Rewarded stimuli / reward aligned') 
         # plt.close()

         # colormap = ['blue', 'red']
         # savefigname = 'SessionComparison-mean-stimuli_' + str(duration[0]) + 'sec'
         # analysis_params = ['Left','Right']
         # pfun.lineplot_sessions(dffTrace_mean_stimuli_baseline, analysis_params, colormap,
         #                      duration, zscoreRun, savefigname, subfolder_path, baseline_subtract,
         #                      title=f'{title_prefix} - Stimuli side / stimulus aligned') 
         # plt.close()

         # print('Completed')
         info.recordingList.loc[ind,'analysisVariableExtracted'] = 1
      except Exception as e:
         info.recordingList.loc[ind,'analysisVariableExtracted'] = 0
         print(f">>>>>>>>>>>>>>>>>>>>>>>  Error occurred while processing {info.recordingList.sessionName[ind]}: {e}")
# Add a line percentage of completion for the for loop
      print(f'Completed {ind+1}/{len(info.recordingList)}')

print( f'All sessions completed. {np.sum(info.recordingList["analysisVariableExtracted"])} out of {len(info.recordingList)} sessions extracted successfully.')


In [ ]:
#lets save the info for later use
repo_analysis_path = r'C:/Users/Huriye/Documents/code/sideBiasLateralisation/analysis'
pickle.dump(info, open(os.path.join(repo_analysis_path, 'sideBiasLateralisation_info.pkl'), 'wb'))

